## Heart Section

Dataset Link : https://www.kaggle.com/datasets/noeyislearning/framingham-heart-study

In [40]:
import pandas as pd

In [41]:
df = pd.read_csv("/content/drive/MyDrive/Datasets/Medical_Disease_NN/framingham_heart_study.csv")

In [42]:
df.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4240 entries, 0 to 4239
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4240 non-null   int64  
 1   age              4240 non-null   int64  
 2   education        4135 non-null   float64
 3   currentSmoker    4240 non-null   int64  
 4   cigsPerDay       4211 non-null   float64
 5   BPMeds           4187 non-null   float64
 6   prevalentStroke  4240 non-null   int64  
 7   prevalentHyp     4240 non-null   int64  
 8   diabetes         4240 non-null   int64  
 9   totChol          4190 non-null   float64
 10  sysBP            4240 non-null   float64
 11  diaBP            4240 non-null   float64
 12  BMI              4221 non-null   float64
 13  heartRate        4239 non-null   float64
 14  glucose          3852 non-null   float64
 15  TenYearCHD       4240 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 530.1 KB


In [44]:
feature = df.drop("TenYearCHD", axis=1)
target = df["TenYearCHD"]

In [45]:
feature.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4240 entries, 0 to 4239
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4240 non-null   int64  
 1   age              4240 non-null   int64  
 2   education        4135 non-null   float64
 3   currentSmoker    4240 non-null   int64  
 4   cigsPerDay       4211 non-null   float64
 5   BPMeds           4187 non-null   float64
 6   prevalentStroke  4240 non-null   int64  
 7   prevalentHyp     4240 non-null   int64  
 8   diabetes         4240 non-null   int64  
 9   totChol          4190 non-null   float64
 10  sysBP            4240 non-null   float64
 11  diaBP            4240 non-null   float64
 12  BMI              4221 non-null   float64
 13  heartRate        4239 non-null   float64
 14  glucose          3852 non-null   float64
dtypes: float64(9), int64(6)
memory usage: 497.0 KB


In [46]:
target.info()

<class 'pandas.core.series.Series'>
RangeIndex: 4240 entries, 0 to 4239
Series name: TenYearCHD
Non-Null Count  Dtype
--------------  -----
4240 non-null   int64
dtypes: int64(1)
memory usage: 33.3 KB


In [47]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    feature,
    target,
    test_size=0.2,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

In [48]:
print(X_train.shape)
print(X_temp.shape)
print(X_val.shape)
print(X_test.shape)

#80% of X_train, 20% X_temp, 10% X_val and 10% X_test

(3392, 15)
(848, 15)
(424, 15)
(424, 15)


In [49]:
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

num_cols = X_train.select_dtypes(exclude=["object"]).columns

num_pipeline = make_pipeline(SimpleImputer(strategy = "median"), StandardScaler())

preprocess = ColumnTransformer(
    [
        ("num", num_pipeline, num_cols)
    ]
)

inputs = preprocess.fit_transform(X_train)
vals = preprocess.transform(X_val)
test = preprocess.transform(X_test)

In [50]:
import tensorflow as tf
import numpy as np

In [52]:
class Heart_disease(tf.keras.Model):
  def __init__(self,units=30,activation="relu", **kwargs):
    super().__init__(**kwargs)
    self.input_wide_norm = tf.keras.layers.Normalization()
    self.input_deep_norm = tf.keras.layers.Normalization()
    self.hidden1 = tf.keras.layers.Dense(units, activation=activation)
    self.hidden2 = tf.keras.layers.Dense(units, activation=activation)
    self.hidden3 = tf.keras.layers.Dense(units, activation=activation)
    self.output_layer = tf.keras.layers.Dense(1, activation="sigmoid")

  def call(self, inputs):
    input_wide, input_deep = inputs
    norm_wide = self.input_wide_norm(input_wide)
    norm_deep = self.input_deep_norm(input_deep)
    hidden1 = self.hidden1(norm_deep)
    hidden2 = self.hidden2(hidden1)
    hidden3 = self.hidden3(hidden2)
    concat = tf.keras.layers.concatenate([norm_wide, hidden3])
    output_layer = self.output_layer(concat)
    return output_layer

model = Heart_disease(30,activation="relu", name = "heart_disease_model")
model.input_wide_norm.adapt(inputs)
model.input_deep_norm.adapt(inputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint("my_checkpoints.weights.h5", save_weights_only=True)

early_stopping_cb = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history = model.fit(
    (inputs, inputs),
    y_train,
    validation_data=((vals, vals), y_val),
    epochs=20,
    callbacks=[checkpoint_cb, early_stopping_cb]
)

Epoch 1/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.8154 - auc: 0.5493 - loss: 0.4790 - val_accuracy: 0.8514 - val_auc: 0.6311 - val_loss: 0.4043
Epoch 2/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8467 - auc: 0.6845 - loss: 0.3999 - val_accuracy: 0.8467 - val_auc: 0.6682 - val_loss: 0.3980
Epoch 3/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8511 - auc: 0.7169 - loss: 0.3891 - val_accuracy: 0.8538 - val_auc: 0.6691 - val_loss: 0.3955
Epoch 4/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8529 - auc: 0.7326 - loss: 0.3806 - val_accuracy: 0.8467 - val_auc: 0.6624 - val_loss: 0.4010
Epoch 5/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8505 - auc: 0.7402 - loss: 0.3781 - val_accuracy: 0.8396 - val_auc: 0.6688 - val_loss: 0.4012
Epoch 6/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8535 - auc: 0.7472 - loss: 0.3738 - val_accuracy: 0.8349 - val_auc: 0.6677 - val_loss: 0.4047
Epoch 7/20
106/106 ━━━━━━━━━━━━━━━━━━━

In [ ]:
%pip